# Member 3 - Pipeline B: pose keypoints (LSTM + Transformer)
MediaPipe Holistic keypoints (body + both hands) -> **Pose-LSTM** (attention pooling) and **Pose-Transformer** (CLS token).
The keypoints are already extracted (`extract_keypoints.py`), so this notebook needs **no MediaPipe** and only a small
data upload (~20 MB: `processed/splits/` + `processed/keypoints/`).

**Use a GPU runtime** (Runtime -> Change runtime type -> T4), although these models are small and also run on CPU.
Training resumes automatically after a disconnect.

## 0. Setup (run every new session)
**Step 0a - Google Drive.** Data and results live on Drive so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Step 0b - the code.** Choose ONE way:
* **`USE_GITHUB = False`** (default for now): the code is not pushed yet -> upload `isl_pose_code.zip` to the top of *My Drive* and it is unzipped here.
* **`USE_GITHUB = True`**: clone the repo. Only use it once `BRANCH` has been pushed to GitHub with the pose code.

In [ ]:
import os, sys
%cd /content
USE_GITHUB = False               # True only after the branch is pushed to GitHub
REPO = "https://github.com/Rani553/indian-sign-language-video-recognition.git"
BRANCH = "feature/pose-models"   # use "develop" once this work is merged
CODE_ZIP = "/content/drive/MyDrive/isl_pose_code.zip"
REPO_DIR = "/content/indian-sign-language-video-recognition"

if USE_GITHUB:
    if not os.path.exists(REPO_DIR):
        !git clone -q $REPO $REPO_DIR
    %cd $REPO_DIR
    !git fetch -q && git checkout -q $BRANCH && git pull -q
    !git log --oneline -3
else:
    assert os.path.exists(CODE_ZIP), f"Upload isl_pose_code.zip to the top of My Drive first ({CODE_ZIP} not found)"
    !rm -rf $REPO_DIR
    !mkdir -p $REPO_DIR
    !unzip -q -o $CODE_ZIP -d $REPO_DIR
    %cd $REPO_DIR

# Forget any 'src' package Python imported earlier (e.g. from a previous, wrong checkout).
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
sys.path.insert(0, REPO_DIR)

assert os.path.exists("src/utils/config.py"), (
    "The code is incomplete: src/utils/config.py is missing. "
    "If USE_GITHUB is True the GitHub branch may not have the code yet - use USE_GITHUB = False.")
print("code OK in", os.getcwd(), "->", sorted(os.listdir(".")))

In [ ]:
# Small extra packages (torch is already installed on Colab). MediaPipe is NOT needed here.
!bash scripts/setup_colab.sh

## 1. Put the data in place
Upload **`isl_pose_data.zip`** to the top of *My Drive* (it contains `processed/splits/` and `processed/keypoints/`).
This cell unzips it into `isl_data/` on Drive the first time and does nothing afterwards.

In [ ]:
import glob, zipfile
from src.utils.config import load_config, resolve_paths
paths = resolve_paths(load_config("configs/pose_lstm.yaml"))
DATA_ZIP = "/content/drive/MyDrive/isl_pose_data.zip"

if len(glob.glob(str(paths["keypoints_dir"] / "*.npz"))) < 958:
    assert os.path.exists(DATA_ZIP), f"Upload isl_pose_data.zip to My Drive first ({DATA_ZIP} not found)"
    paths["data_root"].mkdir(parents=True, exist_ok=True)
    zipfile.ZipFile(DATA_ZIP).extractall(paths["data_root"])
    print("unzipped into", paths["data_root"])

paths = resolve_paths(load_config("configs/pose_lstm.yaml"))
for k in ("splits_dir", "keypoints_dir"):
    print(k, paths[k], "| exists:", paths[k].exists())
print("keypoint files:", len(glob.glob(str(paths["keypoints_dir"] / "*.npz"))), "(expected 958)")

In [ ]:
# Sanity check: split sizes and the model input size.
import pandas as pd
from src.features.keypoint_features import feature_dim
cfg = load_config("configs/pose_lstm.yaml")
print({s: len(pd.read_csv(paths["splits_dir"] / f"{s}.csv")) for s in ("train", "val", "test")}, "(expected 684 / 137 / 137)")
print("feature dim F =", feature_dim(cfg))

## 2. Quick check (2 epochs, tiny subset)
Run this first. If it works, the full runs will too.

In [ ]:
!python -m src.training.train_pose --config configs/pose_lstm.yaml --tiny

## 3. Main models
Baseline: pose + both hands, wrist-relative hand shape, velocity, augmentation, 32 frames. Seed 42, as in Pipeline A.

In [ ]:
!python -m src.training.train_pose --config configs/pose_lstm.yaml

In [ ]:
!python -m src.training.train_pose --config configs/pose_transformer.yaml

In [ ]:
from IPython.display import Image, display
for name in ("pose_lstm", "pose_transformer"):
    display(Image(str(paths["results_root"] / name / "curves.png")))

## 4. Ablations
One change at a time, for both models: hands only, body only, no velocity, no augmentation, 16 frames, mirror.
Each run resumes, so re-run this cell after a disconnect. Pick groups: `!bash scripts/run_ablations_pose.sh parts velocity`

In [ ]:
!bash scripts/run_ablations_pose.sh

## 5. Collect the results
Best validation accuracy of every run (history.csv). The test-set table is made later, once, on the final models
(Member 4's `evaluate` / `compare`).

In [ ]:
import pandas as pd
rows = []
for h in sorted(paths["results_root"].glob("pose_*/history.csv")):
    if h.parent.name.endswith("_tiny"):
        continue
    d = pd.read_csv(h)
    rows.append({"experiment": h.parent.name, "best_val_acc": d["val_acc"].max(),
                 "best_val_top5": d.loc[d["val_acc"].idxmax(), "val_top5"],
                 "best_epoch": int(d["val_acc"].idxmax()), "epochs_run": len(d),
                 "sec_per_epoch": d["epoch_time_s"].mean()})
table = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).round(4)
table.to_csv(paths["results_root"] / "pipeline_b_ablation.csv", index=False)
table

## 6. Share the checkpoints
Copies `best.pt` + `config.yaml` of the two main runs into one Drive folder you can share with the team.
**Do not commit `.pt` files to Git.**

In [ ]:
import shutil
dst = paths["results_root"] / "pose_checkpoints"
dst.mkdir(exist_ok=True)
for name in ("pose_lstm", "pose_transformer"):
    for f in ("best.pt", "config.yaml"):
        shutil.copy(paths["results_root"] / name / f, dst / f"{name}_{f}")
print(sorted(p.name for p in dst.iterdir()), "->", dst)